# Dealing with EPIC Out-of-Time Events -- Part 2: Spectra
<hr style="border: 2px solid #fadbac" />

- **Description:** This thread will allow the user to create a spectrum cleaned from out-of-time events.
- **Level:** Intermediate
- **Data:** XMM observation of the Circinus Galaxy (obsid=0111240101)
- **Requirements:** Must be run using pySAS version 2.2.8 or higher.
- **Credit:** Ryan Tanner (December 2025)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 5 December 2025, for SAS v22.1 and pySAS v2.0

<hr style="border: 2px solid #fadbac" />

## 1. Introduction

In EPIC imaging modes, photons are not only registered during the actual integration interval, but also during the readout of a CCD. These so called Out-of-Time (OoT) events are assigned incorrect RAWY values, leading to a wrong energy correction. OoT events broaden the spectral features, and create a strip of events with wrongly reconstructed position. The fraction of OoT events scales with the (mode-dependent) ratio of integration and readout time, and is highest for pn Full Frame (6.3%) and Extended Full Frame (2.3%) modes (the user is referred to the [XMM-Newton Users Handbook](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/uhb/epicoot.html) for more details). Examples of the effects of OoT events in pn images and spectra are shown in the [User Guide to the XMM-Newton Science Analysis System](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/epicOoT.html). It is important to stress that for most targets a correction of OoT events in the spectrum is not necessary. In any case, a correction is **only necessary** if OoT events overlap the source being investigated.

The general process for dealing with out-of-time contamination is:

1. Calibrate and filter a normal event list.
2. Process an out-of-time event list using the same method.
3. Scale the out-of-time event list based on the out-of-time fraction.
4. Subtract the scaled out-of-time events from the normal event list.

### Chains vs. Procs (`epchain` vs. `epproc`)

In other tutorials we use the "procs" (`epproc` and `emproc`) to generate calibrated event lists. In this tutorial we use the "chains" (`epchain` and `emchain`). The chains do all the same things as the procs, plus more. The filenames for event lists generated by the chains is different than the filenames generated by the procs. pySAS can automatically detect either type of event list.

Event lists generated by the procs have the following general filename structure:

`YYYY_XXXObsIDXX_INST_SSSS_ImagingEvts.ds`

where

- YYYY: The revolution number
- XXXObsIDXX: The observation identifier
- INST: The instrument name, EPN or EMOS1 or EMOS2
- SSSS: The exposure identifier. For instance: S001

Event lists generated by the chains have the following general filename structure:

`PXXXObsIDXXZZSSSSZZEVLI0000.FIT`

where

- XXXObsIDXX: The observation identifier
- ZZ: The instrument name, PN or M1 or M2
- SSSS: The exposure identifier. For instance: S001
- ZZ: An identifier depending on the EPIC mode, and/or whether `withoutoftime` and `withctisrcpos` are set

When either the procs or the chains are run they will silently overwrite any previously generated event lists.

#### SAS Tasks to be Used

- `epchain`[(Documentation for epchain)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/epchain/index.html "epchain Documentation")

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

## 2. Setup

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Useful imports
import re, shutil

# HEASoftpy import
import heasoftpy as hsp

# Importing PyXSPEC
import xspec

# Imports for plotting
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.io import fits
from astropy.wcs import WCS
from regions import CircleSkyRegion
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
obsid  = '0111240101'
my_obs = pysas.ObsID(obsid)

***
The cell below contains a number of functions that will be used throughout this notebook.

In [ ]:
def filter_event_list(in_event_list,
                      out_event_list = 'filtered_event_list.fits',
                      pi_min  = 200,
                      pi_max  = 13000,
                      pattern = None):

    with fits.open(in_event_list) as hdu:
        instrument = hdu[0].header['INSTRUME']

    if instrument == 'EPN':
        filter = 'XMMEA_EP'
        if pattern is None: pattern = 4
    elif 'EMOS' in instrument:
        filter = 'XMMEA_EM'
        if pattern is None: pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              "expression"      : expression, 
              'filteredset'     : out_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes', 
              'updateexposure'  : 'yes', 
              'filterexposure'  : 'yes'}
    
    MyTask('evselect', inargs).run()

def plot_region(image_file, ra, dec, radius, vmin=1.0, vmax=1000.0, zoom = 20.0):
    
    # Define region
    center = SkyCoord(ra, dec)
    region = CircleSkyRegion(center, radius)
    
    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)
    instrument = hdu.header['INSTRUME']

    # Convert region to artist object
    pixel_region = region.to_pixel(wcs)
    artist = pixel_region.as_artist(color='lime')

    # Set image limits
    # This sets the bounds of the lower left (ll) and upper right (ur) of the plot.
    # NOTE: The calculation for the ll and ur of RA is reversed from the
    # calculation for the ll and ur of the DEC (+,- vs. -,+).
    # This preserves the correct orientation of the image.
    # The limits for the RA are also double the limits for the DEC to preserve
    # the aspect ratio.
    ra_ll  = ra+2*zoom*radius
    ra_ur  = ra-2*zoom*radius
    dec_ll = dec-zoom*radius
    dec_ur = dec+zoom*radius
    ra_lim  = [ra_ll.value, ra_ur.value]
    dec_lim = [dec_ll.value, dec_ur.value]
    # The third value "0" sets the "origin", or the index of the first pixel value.
    # It is "0" because Python starts counting at "0".
    (xmin, xmax), (ymin, ymax) = wcs.all_world2pix(ra_lim, dec_lim, 0)

    # Plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.add_artist(artist)
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.title(f'{instrument} Image with Region')
    plt.colorbar()
    plt.show()

def plot_zoom_in(image_file, zoom=4, x=None, y=None, vmin=1.0, vmax=10.0):
    # Open file
    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)
    instrument = hdu.header['INSTRUME']
    im_shape = hdu.shape
    if x is None:
        x_center = int(im_shape[0]/2)
    else:
        x_center = x
    if y is None:
        y_center = int(im_shape[1]/2)
    else:
        y_center = y
    
    # Define the zoomed-in region
    xmin, xmax = x_center-int(x_center/(zoom)), x_center+int(x_center/(zoom))
    ymin, ymax = y_center-int(y_center/(zoom)), y_center+int(y_center/(zoom))

    # Plot
    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
    ax.set_facecolor("black")
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.title(f'{instrument} Image')
    plt.colorbar()
    plt.show()

def plot_spectrum(spectrum,plot_file_name='spectrum_plot.png'):
    xspec.Plot.device='/null'
    xspec.Plot.xAxis = 'keV'

    # Pull off data for main plot
    xspec.Plot('data')
    energy = xspec.Plot.x()
    counts = xspec.Plot.y()
    xErrs = xspec.Plot.xErr()
    yErrs = xspec.Plot.yErr()

    # Get bin edges for "stairs" plot
    bin_edges = []
    for i in spectrum.energies: bin_edges.append(i[0])
    bin_edges.append(spectrum.energies[-1][1])

    # Make the figure and two subplots
    fig, ax0 = plt.subplots(figsize=(9, 7))

    # Main plot
    ax0.errorbar(energy, counts, yerr=yErrs, xerr=xErrs, linestyle='', marker='')
    ax0.set_xscale('log')
    ax0.set_yscale('log')
    ax0.set_xlim([bin_edges[0], bin_edges[-1]])
    ax0.tick_params(top=True,axis="x",direction="in",which='both')
    ax0.tick_params(axis="y",direction="in",which='both',right=True)
    ax0.set_ylabel('counts sec$^{-1}$ keV$^{-1}$')
    ax0.set_title('Data')

The next cell will set up filenames that will be used throughout this notebook.

In [ ]:
filtered_evtli_file  = 'PN_clean_event_list.fits'
filtered_oot_file    = 'PN_clean_outoftime.fits'
pn_obs_image         = 'PN_observation_image.fits'
source_spectrum_file = 'PN_source_spectrum.fits'
oot_spectrum_file    = 'PN_oot_spectrum.fits'
grouped_spectrum     = 'PN_grouped_spectrum.fits'
rmf_file             = 'PN_clean_rmf.fits'
arf_file             = 'PN_clean_arf.fits'
grouped_spectrum_oot = 'PN_grouped_spectrum_oot.fits'

source_spectrum_file_original = 'PN_source_spectrum_original.fits'
grouped_spectrum_original     = 'PN_grouped_spectrum_original.fits'
oot_spectrum_file_original    = 'PN_oot_spectrum_original.fits'
grouped_spectrum_oot_original = 'PN_grouped_spectrum_oot_original.fits'

## 3. Run 'epchain'

This follows the same process as in Part 1 dealing with images, but then moves into extracting the source spectrum.

We will run `epchain` twice. Once to generate a standard event list, and then again to generate the out of time event list.

<div class="alert alert-block alert-info">
    <b>Note:</b> You can use either <tt>basic_setup</tt> or <tt>MyTask</tt> to run <tt>epchain</tt>. Internally <tt>basic_setup</tt> uses <tt>MyTask</tt> to run <tt>epchain</tt>, but it includes some checks to see if it has already been run and will not run it again unless the option '<tt>rerun=True</tt>' is passed into <tt>basic_setup</tt>. Using <tt>MyTask</tt> will silently overwrite any previously generated event lists.
</div>

In [ ]:
inargs = {'runatthkgen'   : False, 
          'runepframes'   : False, 
          'runbadpixfind' : False,
          'runbadpix'     : False}

my_obs.basic_setup(overwrite    = False,
                   rerun        = True,
                   run_epchain  = True,
                   epchain_args = inargs,
                   run_emproc   = False,
                   run_rgsproc  = False)

This will run `epchain` with `withoutoftime=True`. This will **not** overwrite the previously generated event lists because the event lists made with `withoutoftime` have a slightly different file name (see the explanation in the introduction).

In [ ]:
inargs = {'runbackground'    : False, 
          'keepintermediate' : 'raw', 
          'withoutoftime'    : True}

MyTask('epchain', inargs).run()

my_obs.find_event_list_files(print_output=False)

Now let's look at the pn event lists we have.

In [ ]:
for file in my_obs.files['PNevt_list']: print(file)

The filenames of the event lists generated by the first run of `epchain` will look like this:

P0111240101PNS003**PI**EVLI0000.FIT

The filenames of the event lists generated by the second run of `epchain`, with `withoutoftime=True`, will look like this:

P0111240101PNS003**OO**EVLI0000.FIT

They are the exact same except the first has "**PI**" in the name, and the second has "**OO**" in the name. The next cell will store the filenames in the variables `event_list_file` and `outoftime_file`.

In [ ]:
for filename in my_obs.files['PNevt_list']:
    if re.search('.*PN.*PIEVLI.*FIT$',filename):
        event_list_file = filename
    if re.search('.*PN.*OOEVLI.*FIT$',filename):
        outoftime_file = filename

Now we do some basic filtering to make clean event lists.

In [ ]:
filter_event_list(event_list_file, out_event_list=filtered_evtli_file)

In [ ]:
filter_event_list(outoftime_file, out_event_list=filtered_oot_file)

## 4. Extract Source Spectrum

### 4.1 Identify the Source

We begin by generating an image so that we can locate the source we need. The source is contaminated by out-of-time events.

In [ ]:
# "Normal" event list
my_obs.quick_eplot(filtered_evtli_file, image_file=pn_obs_image, vmin=10.0, vmax=1000.0)

The out-of-time events from the bright source in the center create the stripe that is visible going down from the source. Let's zoom in on a source in this stripe.

In [ ]:
plot_zoom_in(pn_obs_image, zoom=3, x=320, y=240, vmin=10.0, vmax=1000.0)

Let's select the source that has been contaminated by out-of-time events. (Note: we are making the radius of the region larger than is necessary. This is to exaggerate the effects of the out-of-time events.)

In [ ]:
source_RA  = 213.255 * u.deg # degrees
source_Dec = -65.4305 * u.deg # degrees
source_rad = 40.0 * u.arcsec # arcseconds

plot_region(pn_obs_image, source_RA, source_Dec, source_rad, vmin=10.0, vmax=1000.0, zoom = 10)

### 4.2 Extract the Source Spectrum

We will do this twice, first extracting the source from the clean event list, and then again from the out-of-time event list.

In [ ]:
# Clean Event List
source_region = "CIRCLE({0},{1},{2})".format(source_RA.value,source_Dec.value,source_rad.to(u.deg).value)
expression = "'(FLAG==0) && (PATTERN<=4) && ((RA,DEC) in {0})'".format(source_region)

inargs = {'table'           : filtered_evtli_file,
          'energycolumn'    : 'PI',
          'filtertype'      : 'expression',
          'expression'      : expression,
          'withspectrumset' : 'yes',
          'spectrumset'     : source_spectrum_file,
          'spectralbinsize' : '5',
          'specchannelmin'  : '0',
          'specchannelmax'  : 20479}

MyTask('evselect', inargs).run()

Because everything else stays the same we can reuse the `inargs` dictionary and just change the names of the input and output files.

In [ ]:
# Out-of-time Event List
inargs['table'] = filtered_oot_file
inargs['spectrumset'] = oot_spectrum_file

MyTask('evselect', inargs).run()

We will generate the `rmf` and `arf` to use later. We will use the same `rmf` and `arf` for all our spectra since they all come from the same region.

In [ ]:
NBINS = 1490

inargs = {'rmfset'         : rmf_file,
          'spectrumset'    : source_spectrum_file,
          'withenergybins' : 'yes',
          'energymin'      : 0.1,
          'energymax'      : 15,
          'nenergybins'    : NBINS}

MyTask('rmfgen', inargs).run()


inargs = {'arfset'         : arf_file,
          'spectrumset'    : source_spectrum_file,
          'withrmfset'     : 'yes',
          'rmfset'         : rmf_file,
          'withbadpixcorr' : 'yes',
          'badpixlocation' : filtered_evtli_file,
          'setbackscale'   : 'yes'}

MyTask('arfgen', inargs).run()

Because we will be making some changes to the spectrum files in the next section we will make some grouped spectra using the original unmodified files so we can compare the final result to the unmodified spectrum.

<div class="alert alert-block alert-info">
    <b>Note:</b> We are grouping the spectra into regular bins with the same number of channels (5) per bin. This is only for comparison between different spectra.
</div>

In [ ]:
inargs = {'spectrumset' : source_spectrum_file,
          'groupedset'  : grouped_spectrum_original,
          'arfset'      : arf_file,
          'rmfset'      : rmf_file,
          'regbinstart' : 1,
          'regbinend'   : NBINS,
          'regbinwid'   : 5}

MyTask('specgroup', inargs).run()


inargs = {'spectrumset' : oot_spectrum_file,
          'groupedset'  : grouped_spectrum_oot_original,
          'arfset'      : arf_file,
          'rmfset'      : rmf_file,
          'regbinstart' : 1,
          'regbinend'   : NBINS,
          'regbinwid'   : 5}

MyTask('specgroup', inargs).run()

### 4.3 Find the Difference

Now we need to rearrange things using HEASoftpy. First we change the name of the `COUNTS` column in the out-of-time event spectrum to `CTS_OOT`.

<div class="alert alert-block alert-info">
    <b>Note:</b> This will modify the original and out-of-time spectrum files.
</div>

In [ ]:
hsp.fparkey(value    = 'CTS_OOT',
            fitsfile = oot_spectrum_file+'+1',
            keyword  = 'TTYPE2')

Then copy the `CTS_OOT` column of the out-of-time event spectrum into the source spectrum.

In [ ]:
hsp.faddcol(infile  = source_spectrum_file+'+1',
            colfile = oot_spectrum_file+'+1',
            colname = 'CTS_OOT')

Multiply the values in the column CTS_OOT by 0.063 (6.3%) because this is the expected fraction of out-of-time events for the pn in Full Frame Mode. The fraction of out-of-time events depends on the camera mode, and is also different for the MOS cameras.

In [ ]:
hsp.fcalc(infile  = source_spectrum_file+'+1',
          outfile = source_spectrum_file,
          clname  = 'CTS_OOT',
          expr    = 'CTS_OOT*0.063',
          clobber = 'yes')

Subtract the rescaled values of the `CTS_OOT` from the `COUNTS` column of the source spectrum.

In [ ]:
hsp.fcalc(infile  = source_spectrum_file+'+1',
          outfile = source_spectrum_file,
          clname  = 'COUNTS',
          expr    = 'COUNTS-CTS_OOT',
          clobber = 'yes')

Now we can group the modified spectrum file.

In [ ]:
inargs = {'spectrumset' : source_spectrum_file,
          'groupedset'  : grouped_spectrum,
          'arfset'      : arf_file,
          'rmfset'      : rmf_file,
          'regbinstart' : 1,
          'regbinend'   : NBINS,
          'regbinwid'   : 5}

MyTask('specgroup', inargs).run()

Now we can plot the original, uncorrected spectrum, along with the spectrum corrected for out-of-time contamination. The subplot below that shows the difference between the two.

In [ ]:
spectra = [grouped_spectrum,grouped_spectrum_original]

xspec.Plot.device='/null'
xspec.Plot.xAxis = 'keV'

# Make the figure and two subplots
fig, (ax0, ax1) = plt.subplots(nrows=2, sharex=True, height_ratios=[2.5, 1],figsize=(9, 7))

energy = []
counts = []

# Pull off data for main plot
for i,file in enumerate(spectra):
    xspec.AllData.clear()
    spectrum = xspec.Spectrum(file)
    spectrum.ignore('0.0-0.5,10.0-**')
    xspec.Plot('data')
    energy.append(xspec.Plot.x())
    counts.append(xspec.Plot.y())
    xErrs = xspec.Plot.xErr()
    yErrs = xspec.Plot.yErr()
    ax0.errorbar(energy[i], counts[i], yerr=yErrs, xerr=xErrs, linestyle='', marker='')
    
ax0.set_xscale('log')
ax0.set_yscale('log')
ax0.tick_params(top=True,axis="x",direction="in",which='both')
ax0.tick_params(axis="y",direction="in",which='both',right=True)
ax0.set_ylabel('counts sec$^{-1}$ keV$^{-1}$')
ax0.set_title('Data')
ax0.grid(which='minor')

diff = np.array(counts[0]) - np.array(counts[1])

# Ratio plot
ax1.plot(energy[0], diff, linestyle='', marker='o')
ax1.set_xscale('log')
ax1.tick_params(top=True,axis="x",direction="in",which='both')
ax1.tick_params(axis="y",direction="in",which='both')
ax1.set_xlabel('Energy (keV)')
ax1.set_ylabel('Difference')
ax1.grid(which='minor')

# This puts the plots together with no space in between
plt.subplots_adjust(hspace=.0)

This will plot the (unscaled) out-of-time events. This shows the spectral shape of the out-of-time contamination.

In [ ]:
xspec.AllData.clear()
spectrum = xspec.Spectrum(grouped_spectrum_oot_original)
spectrum.ignore('0.0-0.5,10.0-**')
plot_spectrum(spectrum)